# CS-808 Lecture 5 — Reproducible pandas pipelines
## The same inflation investigation, continued

Lecture 4 V3 acquired the evidence — World Bank annual inflation and exchange rates, FRED daily Brent oil,
PBS monthly CPI, Apple/SBP iPhone inputs — and persisted it under `data/evidence/` in this student package. Today:

> **We have downloaded the evidence. How do we transform it into reliable results without changing its meaning?**

Every method block follows: **question → prediction → short code → result → interpretation → validation**.
Offline; the only inputs are the Lecture 4 V3 evidence pack and the synthetic failure laboratory.

In [2]:
# Load the published lecture inputs in Colab; verify local inputs offline.
from pathlib import Path
import hashlib, sys
from urllib.request import urlopen
PUBLIC_ROOT = Path.cwd()
_files = {'data/category_lookup.csv': 'e7d09d4a3cc600a8eb0dc78224c2b46fd937dd7291ea231be45a0bc5b4dadecb', 'data/evidence/acquisition_log.json': '0c9584f9d399cefebd53008ae6867767e86ad92b0f7a81f97924e9cfc15de05d', 'data/evidence/evidence_manifest.json': '567c7bdf13ed7490ccc340b26ce966412adbf090087d005c1152d2011a71b961', 'data/evidence/fred_brent_daily_usd.csv': '68b1b12a3e217ed57b60f53cea699e3fad740f714bbaa6d23afb96c8dac94c1e', 'data/evidence/iphone_launch_comparison.csv': '4af85e8132f99c865f836845a4e0e06b06a4ae993c95fcf77cabfea6710a69d2', 'data/evidence/pbs_cpi_monthly.csv': 'c1790f9210ab08e204e94078b2f1f58f436e1d6f12a098abf4986cb82c397e04', 'data/evidence/pbs_fresh_fruit_layers.csv': '2827e72adcd956a2c2821c6e45e52fc7c91ff268a00eee05e7119813c74ee065', 'data/evidence/worldbank_pk_annual.csv': 'ec62fedc2f19843669a6ea30942ee243233ed15e0400706cc711feb9c4e13127', 'data/practice_prices_raw.csv': 'bc0d3c29594f0c42d4dec3123cc4d0ffe4e991c7bdf1ca59365fdafa7f8de864', 'methods.py': 'e2022a93ec28f0b92061e48da5da4353d2ed691d5f36c5c3b4c8f283c4b2c0f2'}
_base_url = 'https://raw.githubusercontent.com/DrFarrukh/teaching/main/courses/tools-and-tech-in-data-science/lectures/lecture-05/'
_in_colab = 'google.colab' in sys.modules
for _name, _sha in _files.items():
    _path = PUBLIC_ROOT / _name
    if not _path.exists():
        if not _in_colab:
            raise FileNotFoundError('Extract the student ZIP and run this notebook from its lecture folder. Missing: ' + _name)
        with urlopen(_base_url + _name, timeout=30) as _response:
            _bytes = _response.read()
        if hashlib.sha256(_bytes).hexdigest() != _sha:
            raise ValueError('Downloaded input failed its checksum: ' + _name)
        _path.parent.mkdir(parents=True, exist_ok=True)
        _path.write_bytes(_bytes)
    if hashlib.sha256(_path.read_bytes()).hexdigest() != _sha:
        raise ValueError('Input differs from the published snapshot: ' + _name)
print('Lecture inputs ready; frozen source checksums verified.')

from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

ROOT = PUBLIC_ROOT
sys.path.insert(0, str(ROOT))
PACK = ROOT / 'data' / 'evidence'  # frozen Lecture 4 evidence, unchanged
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 150)
log = json.loads((PACK / 'acquisition_log.json').read_text())
print('pandas', pd.__version__, '| numpy', np.__version__)
pd.DataFrame(log)[['dataset', 'grain', 'unit', 'origin']]

pandas 3.0.6 | numpy 2.5.3


,dataset,grain,unit,origin
0,worldbank_pk_inflation_annual,one row per calendar year,"annual CPI inflation, %",PRESERVED SNAPSHOT retrieved 20261001T100931Z ...
1,worldbank_pk_pkr_per_usd_annual,one row per calendar year,"PKR per USD, period average",PRESERVED SNAPSHOT retrieved 20261001T100935Z ...
2,fred_brent_daily_usd,one row per reported observation date; this ca...,USD per barrel,PRESERVED SNAPSHOT retrieved 20261001T100927Z ...
3,pbs_cpi_monthly,"one row per (indicator, month)","CPI index, 2015-16 = 100",PRESERVED SNAPSHOT retrieved 20261001T100949Z ...


In [3]:
# Verify every persisted input, then link the raw Brent file to the acquisition record.
manifest = json.loads((PACK / 'evidence_manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((PACK / name).read_bytes()).hexdigest() == expected, name
brent_rec = next(e for e in log if e['dataset'] == 'fred_brent_daily_usd')
digest = hashlib.sha256((PACK / 'fred_brent_daily_usd.csv').read_bytes()).hexdigest()
assert digest == brent_rec['sha256_of_raw_bytes']
print('Brent raw bytes verified against the Lecture 4 acquisition log:', digest[:16], '...')

Brent raw bytes verified against the Lecture 4 acquisition log: 68b1b12a3e217ed5 ...


## A. Inspect and parse — what exactly did we download?
**Question:** what do the raw Brent bytes contain? **Predict:** text columns on **reported observation dates**,
not every calendar day. This capture omits weekends and has some blank price tokens.
Weekends/closures are not zero prices; we check the actual tokens rather than assuming coverage.

In [4]:
# read_csv with a deliberate policy: everything is text, no token is silently converted.
raw = pd.read_csv(PACK / 'fred_brent_daily_usd.csv', dtype='string', keep_default_na=False)
print('shape:', raw.shape)
display(raw.head(4))
display(raw.dtypes.to_frame('dtype'))

shape: (10270, 2)


,observation_date,DCOILBRENTEU
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60


,dtype
observation_date,string
DCOILBRENTEU,string


In [5]:
# One bracket -> Series; two brackets -> DataFrame; .loc selects by label, explicitly.
print('Series  :', raw['DCOILBRENTEU'].shape)
print('DataFrame:', raw[['DCOILBRENTEU']].shape)
by_date = raw.set_index('observation_date')
display(by_date.loc[['2026-09-25', '2026-09-28'], ['DCOILBRENTEU']])

Series  : (10270,)
DataFrame: (10270, 1)


,DCOILBRENTEU
observation_date,
2026-09-25,116.01
2026-09-28,119.97


**Parsing policy, stated before parsing:** a **blank** token is *missing* (unreported quote), not zero;
a **malformed** token (not blank, not numeric) is a different event; a **non-positive** price violates the
domain. Three masks, three decisions — never one silent conversion. Absent weekend rows and blank price
tokens are both missingness, but of different origin; neither becomes a price.

In [6]:
missing = raw['DCOILBRENTEU'].eq('')
brent = pd.DataFrame({'date': pd.to_datetime(raw['observation_date'], format='%Y-%m-%d'),
                      'usd_per_barrel': pd.to_numeric(raw['DCOILBRENTEU'].mask(missing), errors='coerce')})
parse_failure = ~missing & brent.usd_per_barrel.isna()
non_positive = brent.usd_per_barrel.le(0).fillna(False)
weekend_rows = brent.date.dt.dayofweek.ge(5).sum()
print('missing quotes:', int(missing.sum()), '| malformed:', int(parse_failure.sum()),
      '| invalid (<=0):', int(non_positive.sum()))
assert not parse_failure.any() and not non_positive.any()
print(f'Interpretation: {int(missing.sum())} blank tokens are absent prices; the cause needs metadata.',
      f'weekend rows: {int(weekend_rows)} — reported dates are not a full calendar, and gaps are not $0 oil.')

missing quotes: 1173 | malformed: 0 | invalid (<=0): 0
Interpretation: 1173 blank tokens are absent prices; the cause needs metadata. weekend rows: 0 — reported dates are not a full calendar, and gaps are not $0 oil.


**The danger of filling unknown values with zero** — on real data, not a toy example.
**Predict:** March 2024 had 31 calendar days. What happens to the monthly mean if the unobserved days
are filled with zero?

In [7]:
mar = brent.loc[brent.date.dt.to_period('M').eq(pd.Period('2024-03'))]
print('March 2024: rows', len(mar), '| observed prices', int(mar.usd_per_barrel.notna().sum()),
      '| calendar days', int(mar.date.dt.days_in_month.iloc[0]))
print('mean over observed days   :', round(mar.usd_per_barrel.mean(), 2))
calendar = mar.set_index('date').usd_per_barrel.reindex(pd.date_range('2024-03-01', '2024-03-31'))
print('mean with gaps zero-filled:', round(calendar.fillna(0).mean(), 2), '  <- wrong: invents $0 oil')
print('Validation: the denominator changed the answer, not the data. Unknown stays unknown.')

March 2024: rows 21 | observed prices 20 | calendar days 31
mean over observed days   : 85.41
mean with gaps zero-filled: 55.1   <- wrong: invents $0 oil
Validation: the denominator changed the answer, not the data. Unknown stays unknown.


## B. Data quality — grain, keys, quarantine
**Question:** what identifies one PBS observation? **Predict:** `(indicator, month)` — the composite key —
should be unique, because Lecture 4's audit said so. Trust, then verify.

In [8]:
pbs = pd.read_csv(PACK / 'pbs_cpi_monthly.csv')
print('rows:', len(pbs), '| series:', pbs.indicator.nunique(),
      '| months per series:', pbs.groupby('indicator').size().unique().tolist())
assert not pbs.duplicated(['indicator', 'month']).any()      # composite key is unique
assert pbs.index_2015_16_100.gt(0).all()                      # index levels are positive
print('Grain confirmed: one row per (indicator, month); base 2015-16 = 100.')

rows: 650 | series: 13 | months per series: [50]
Grain confirmed: one row per (indicator, month); base 2015-16 = 100.


**Duplicates come in two kinds.** A *confirmed copy* (same measurement, recorded twice) may be removed — with a
log entry. *Conflicting measurements* (same key, different values) are **quarantined**, never averaged and never
resolved by keeping the first row. Demonstrated on a **clearly labelled defect copy** — raw files are never
touched.

In [9]:
demo = brent.loc[brent.date.dt.year.eq(2026)].copy()
conflict = demo.loc[demo.date.eq(pd.Timestamp('2026-09-29'))].assign(usd_per_barrel=999.0)
demo = pd.concat([demo, conflict], ignore_index=True)        # labelled defect copy
dup = demo.date.duplicated(keep=False)
quarantined, clean_demo = demo.loc[dup], demo.loc[~dup]
print('conflicting rows quarantined:', len(quarantined), '(not averaged, not first-kept)')
display(quarantined[['date', 'usd_per_barrel']])
issue_log = pd.DataFrame([{'table': 'brent_demo', 'key': '2026-09-29', 'issue': 'conflicting_measurement',
                           'action': 'quarantine_both', 'reason': 'same date, two values; source must resolve'}])
display(issue_log)

conflicting rows quarantined: 2 (not averaged, not first-kept)


,date,usd_per_barrel
193,2026-09-29,113.96
194,2026-09-29,999.0


,table,key,issue,action,reason
0,brent_demo,2026-09-29,conflicting_measurement,quarantine_both,"same date, two values; source must resolve"


In [10]:
# The pack file itself is untouched; the defect lives only in the labelled in-memory copy.
digest = hashlib.sha256((PACK / 'fred_brent_daily_usd.csv').read_bytes()).hexdigest()
assert digest == brent_rec['sha256_of_raw_bytes']
print('Raw evidence file unchanged. Defects for teaching are injected into copies, never into sources.')

Raw evidence file unchanged. Defects for teaching are injected into copies, never into sources.


### The Data Quality Failure Laboratory (synthetic — preserved from V2)
Official files rarely show every failure mode, so we keep a controlled 15-record laboratory. **These are
invented index-like values, not CPI.** Documented policies: row 3 is a *confirmed* accidental copy of row 2;
blank / `NA` / `not reported` are missing tokens; commas are thousands separators; the negative value is
invalid; `1O20` contains a letter O (parse failure, do not "fix" to zero); `31/02/2024` is an invalid date;
rows 8 and 12 conflict on Housing-January; row 13's zero is a real observation.

In [11]:
lab = pd.read_csv(DATA / 'practice_prices_raw.csv',
                  dtype={'category_text': 'string', 'date_text': 'string', 'value_text': 'string'},
                  keep_default_na=False)
lab_before = lab.copy(deep=True)
print('15 SYNTHETIC records:')
display(lab)

15 SYNTHETIC records:


,source_row,category_text,date_text,value_text
0,1,food,01/01/2024,100
1,2,food,01/02/2024,110
2,3,FOOD,01/02/2024,110
3,4,FOOD,01/03/2024,121
4,5,transport,01/01/2024,200
5,6,TRANSPORT,01/02/2024,
6,7,TRANSPORT,01/03/2024,-5
7,8,housing,01/01/2024,"1,000"
8,9,HOUSING,01/02/2024,not reported
9,10,HOUSING,01/03/2024,1O20


In [12]:
# Normalize and parse with explicit failure masks (the same three-event policy as the Brent data).
work = lab.copy(deep=True)
work['category'] = work['category_text'].str.strip().str.upper()
work['date'] = pd.to_datetime(work['date_text'].str.strip(), format='%d/%m/%Y', errors='coerce')
text = work['value_text'].str.strip().str.replace(',', '', regex=False)
source_missing = text.str.upper().isin(['', 'NA', 'NOT REPORTED'])
work['value'] = pd.to_numeric(text.mask(source_missing), errors='coerce').astype('Float64')
work['value_reason'] = 'observed'
work.loc[source_missing, 'value_reason'] = 'source_missing'
work.loc[~source_missing & work['value'].isna(), 'value_reason'] = 'parse_failure'
negative = work['value'].lt(0).fillna(False)
work.loc[negative, 'value_reason'] = 'invalid_negative'
work.loc[negative, 'value'] = pd.NA
display(work.loc[work.value_reason.ne('observed'), ['source_row', 'value_text', 'value', 'value_reason']])

,source_row,value_text,value,value_reason
5,6,,<NA>,source_missing
6,7,-5,<NA>,invalid_negative
8,9,not reported,<NA>,source_missing
9,10,1O20,<NA>,parse_failure


In [13]:
# Remove ONLY the confirmed copy; quarantine the invalid date and BOTH sides of the key conflict.
no_copy = work.loc[work.source_row.ne(3)].copy()
date_quarantine = no_copy.loc[no_copy.date.isna()]
valid = no_copy.loc[no_copy.date.notna()]
conflict = valid.duplicated(['category', 'date'], keep=False)
key_quarantine = valid.loc[conflict]
clean = valid.loc[~conflict].sort_values(['category', 'date']).reset_index(drop=True)
print(f'15 rows -> {len(clean)} clean keys | {len(date_quarantine) + len(key_quarantine)} quarantined',
      f'| 1 confirmed copy removed | observed values: {int(clean.value.count())}')
display(clean[['source_row', 'category', 'date', 'value', 'value_reason']])

15 rows -> 11 clean keys | 3 quarantined | 1 confirmed copy removed | observed values: 7


,source_row,category,date,value,value_reason
0,1,FOOD,2024-01-01,100.0,observed
1,2,FOOD,2024-02-01,110.0,observed
2,4,FOOD,2024-03-01,121.0,observed
3,9,HOUSING,2024-02-01,<NA>,source_missing
4,10,HOUSING,2024-03-01,<NA>,parse_failure
5,13,MISC,2024-01-01,0.0,observed
6,14,MISC,2024-02-01,5.0,observed
7,15,MISC,2024-03-01,10.0,observed
8,5,TRANSPORT,2024-01-01,200.0,observed
9,6,TRANSPORT,2024-02-01,<NA>,source_missing


In [14]:
# Independent validation: the packaged function must reproduce the staged result, and the input must be intact.
from methods import prepare_prices
packaged, issues, lab_quarantine = prepare_prices(lab, confirmed_copies=(3,))
cols = ['source_row', 'category', 'date', 'value', 'value_reason']
assert_frame_equal(packaged[cols], clean[cols])
assert_frame_equal(lab, lab_before)
print('issue events:', len(issues), '| quarantined:', len(lab_quarantine),
      '| staged and packaged computations agree; raw input unmutated')
display(issues)

issue events: 8 | quarantined: 3 | staged and packaged computations agree; raw input unmutated


,source_row,issue,action,reason
0,3,confirmed_copy,remove,Exercise source note confirms an accidental re...
1,6,source_missing,retain_missing,Documented missing token
2,7,invalid_negative,retain_missing,Synthetic exercise requires nonnegative values
3,8,unresolved_key,quarantine,All repeated-key candidates require source rec...
4,9,source_missing,retain_missing,Documented missing token
5,10,parse_failure,retain_missing,Unparseable value; raw token retained
6,11,invalid_date,quarantine,No valid period key can be constructed
7,12,unresolved_key,quarantine,All repeated-key candidates require source rec...


## C. Combining data — concat for batches, merge for relationships
**Question:** if we split PBS into two compatible month-batches and stack them, do we get the original back?
**Predict:** yes — and stacking one batch twice must be *detectable* through duplicated keys.

In [15]:
months = sorted(pbs.month.unique())          # YYYY-MM text sorts chronologically
half = months[len(months) // 2]
batch_a, batch_b = pbs.loc[pbs.month.le(half)], pbs.loc[pbs.month.gt(half)]
stacked = pd.concat([batch_a, batch_b], ignore_index=True)
assert_frame_equal(stacked.sort_values(['indicator', 'month']).reset_index(drop=True),
                   pbs.sort_values(['indicator', 'month']).reset_index(drop=True))
doubled = pd.concat([batch_a, batch_a], ignore_index=True)
print('stacked batches:', len(stacked), '| accidental repeat:', len(doubled),
      '| duplicated keys detected:', int(doubled.duplicated(['indicator', 'month']).sum()))

stacked batches: 650 | accidental repeat: 676 | duplicated keys detected: 338


**The real join of this course:** PBS **monthly** CPI × FRED **daily** Brent. **Question:** can daily rows join
monthly rows directly? **Predict:** no — the grains differ. Aggregate oil to one record per month *first*,
stating the denominator (observed trading days, not calendar days).

In [16]:
brent['period'] = brent.date.dt.to_period('M')
brent_monthly = brent.groupby('period').agg(mean_brent_usd=('usd_per_barrel', 'mean'),
                                            observed_days=('usd_per_barrel', 'count'),
                                            rows=('usd_per_barrel', 'size')).reset_index()
brent_monthly['calendar_days'] = brent_monthly.period.dt.days_in_month
print('monthly oil records:', len(brent_monthly), '| unique months:', not brent_monthly.period.duplicated().any())
print('count = observed prices, size = rows; the mean uses observed days only.')
display(brent_monthly.tail(4).round({'mean_brent_usd': 2}))

monthly oil records: 473 | unique months: True
count = observed prices, size = rows; the mean uses observed days only.


,period,mean_brent_usd,observed_days,rows,calendar_days
469,2026-06,85.4,22,22,30
470,2026-07,83.76,23,23,31
471,2026-08,91.08,20,21,31
472,2026-09,114.08,21,21,30


In [17]:
# Many CPI rows (3 series) relate to ONE oil row per month: many-to-one. Predict 3 x 12 = 36 rows.
sel = pbs.loc[pbs.indicator.isin(['PCPI_IX', 'PCPI_CP_01_IX', 'PCPI_CP_07_IX'])].copy()
sel['period'] = pd.PeriodIndex(sel.month, freq='M')
recent = sel.loc[sel.period.between(pd.Period('2025-09'), pd.Period('2026-08'))]
joined = recent.merge(brent_monthly, on='period', how='left', validate='many_to_one', indicator=True)
assert len(joined) == 36 and joined._merge.eq('both').all()
print('rows:', len(joined), '| unmatched:', int(joined._merge.ne('both').sum()))
display(joined.loc[joined.period.eq(pd.Period('2026-08')),
                   ['month', 'label', 'index_2015_16_100', 'mean_brent_usd', 'observed_days']].round(2))

rows: 36 | unmatched: 0


,month,label,index_2015_16_100,mean_brent_usd,observed_days
11,2026-08,Food and non-alcoholic beverages,313.99,91.08,20
23,2026-08,Transport,382.06,91.08,20
35,2026-08,General,300.50,91.08,20


**Predict:** if the monthly oil table accidentally contains the same month twice, an unchecked merge multiplies
rows — and `validate='many_to_one'` refuses. Matching dates establish **alignment, not causality**.

In [18]:
dup_month = brent_monthly.loc[brent_monthly.period.eq(pd.Period('2026-08'))]
bad = pd.concat([brent_monthly, dup_month], ignore_index=True)   # labelled defect copy
unsafe = recent.merge(bad, on='period', how='left')
print('unchecked join rows:', len(unsafe), '| primary rows:', len(recent),
      '| rows silently added:', len(unsafe) - len(recent))
try:
    recent.merge(bad, on='period', how='left', validate='many_to_one')
except pd.errors.MergeError as e:
    print('validate= rejected it:', str(e)[:90])

unchecked join rows: 39 | primary rows: 36 | rows silently added: 3
validate= rejected it: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:



In [19]:
# indicator=True makes unmatched observations visible instead of silently absent.
audit = recent.merge(brent_monthly.loc[brent_monthly.period.ge(pd.Period('2026-01'))],
                     on='period', how='left', validate='many_to_one', indicator=True)
print(audit._merge.value_counts().to_dict())
display(audit.loc[audit._merge.eq('left_only'), ['month', 'label']].head(4))
print('CPI months with no oil record in the restricted table: reported, not zero-filled.')

{'both': 24, 'left_only': 12, 'right_only': 0}


,month,label
0,2025-09,Food and non-alcoholic beverages
1,2025-10,Food and non-alcoholic beverages
2,2025-11,Food and non-alcoholic beverages
3,2025-12,Food and non-alcoholic beverages


CPI months with no oil record in the restricted table: reported, not zero-filled.


## D. Aggregation — groupby, denominators, weighting
**Inflation rate from index LEVELS:** `yoy = 100 × (I_t / I_{t−12} − 1)`. The denominator is the index
**12 months earlier**, not last month, and the first 12 months of every series have no YoY — a denominator that
does not exist yet.

In [20]:
pbs_sorted = pbs.sort_values(['indicator', 'month']).reset_index(drop=True)
pbs_sorted['yoy_pct'] = pbs_sorted.groupby('indicator')['index_2015_16_100'].pct_change(12, fill_method=None) * 100
general = pbs_sorted.loc[pbs_sorted.indicator.eq('PCPI_IX')]
display(general[['month', 'index_2015_16_100', 'yoy_pct']].tail(5).round(2))
print('Missing YoY for the first 12 months per series:', int(general.yoy_pct.isna().sum()), '(expected: 12)')

,month,index_2015_16_100,yoy_pct
645,2026-04,292.81,10.89
646,2026-05,294.34,11.66
647,2026-06,293.47,11.07
648,2026-07,296.97,9.20
649,2026-08,300.50,11.15


Missing YoY for the first 12 months per series: 12 (expected: 12)


In [21]:
# agg returns one row PER GROUP; transform broadcasts one aggregate back to EVERY row.
brent['year'] = brent.date.dt.year
d26 = brent.loc[brent.year.eq(2026)].copy()
d26['deviation_from_2026_mean'] = d26.usd_per_barrel - d26.groupby('year')['usd_per_barrel'].transform('mean')
agg_view = d26.groupby('period').agg(mean=('usd_per_barrel', 'mean'),
                                     observed=('usd_per_barrel', 'count'),
                                     rows=('usd_per_barrel', 'size')).tail(3)
display(agg_view.round({'mean': 2}))
display(d26[['date', 'usd_per_barrel', 'deviation_from_2026_mean']].tail(4)
        .round({'usd_per_barrel': 2, 'deviation_from_2026_mean': 2}))

,mean,observed,rows
period,,,
2026-07,83.76,23,23
2026-08,91.08,20,21
2026-09,114.08,21,21


,date,usd_per_barrel,deviation_from_2026_mean
10266,2026-09-24,120.92,27.89
10267,2026-09-25,116.01,22.98
10268,2026-09-28,119.97,26.94
10269,2026-09-29,113.96,20.93


In [22]:
# Why group means can differ from the pooled mean: the denominators (trading days per month) differ.
y24 = brent.loc[brent.period.dt.year.eq(2024)]
monthly_means = y24.groupby('period').usd_per_barrel.mean()
print('mean of the 12 monthly means :', round(monthly_means.mean(), 2))
print('pooled mean over observed days:', round(y24.usd_per_barrel.mean(), 2))
print('observed days per month:', [int(x) for x in sorted(y24.groupby('period').usd_per_barrel.count().unique())])
print('hand check: groups [60,70,80] and [90,110] -> mean of means 85, pooled 82; counts 3 vs 2 explain it.')

mean of the 12 monthly means : 80.53
pooled mean over observed days: 80.52
observed days per month: [20, 21, 22, 23]
hand check: groups [60,70,80] and [90,110] -> mean of means 85, pooled 82; counts 3 vs 2 explain it.


## E. Reshaping — long versus wide, with unique cell keys
**Question:** reshape three CPI series into one column per series. **Predict:** `pivot` needs a unique
`(month, series)` key per cell; `melt` reverses it without loss.

In [23]:
wide = recent.pivot(index='period', columns='label', values='index_2015_16_100')
print('wide:', wide.shape, '(one row per month, one column per series)')
display(wide.tail(3).round(1))
long_again = wide.reset_index().melt(id_vars='period', value_name='index_2015_16_100')
same_keys = (set(zip(long_again.label, long_again.period.astype(str)))
             == set(zip(recent.label, recent.month)))
assert same_keys
cols = ['period', 'label', 'index_2015_16_100']
assert_frame_equal(long_again[cols].sort_values(cols[:2]).reset_index(drop=True),
                   recent[cols].sort_values(cols[:2]).reset_index(drop=True), check_dtype=False)
print('melted back:', len(long_again), 'rows | identical keys AND values:', same_keys)

wide: (12, 3) (one row per month, one column per series)


label,Food and non-alcoholic beverages,General,Transport
period,,,
2026-06,296.5,293.5,389.9
2026-07,308.9,297.0,369.5
2026-08,314.0,300.5,382.1


melted back: 36 rows | identical keys AND values: True


In [24]:
# Two records claiming the same cell: pivot refuses; pivot_table silently AVERAGES. Labelled defect copy.
clash = pd.concat([recent, recent.iloc[[0]].assign(index_2015_16_100=999.0)], ignore_index=True)
try:
    clash.pivot(index='period', columns='label', values='index_2015_16_100')
except ValueError as e:
    print('pivot refused the ambiguous cell:', str(e)[:80])
r0 = recent.iloc[0]
silent = clash.pivot_table(index='period', columns='label', values='index_2015_16_100')
print('pivot_table silently averaged the conflict:', round(float(silent.loc[r0.period, r0.label]), 1),
      '| true value:', float(r0.index_2015_16_100))

pivot refused the ambiguous cell: Index contains duplicate entries, cannot reshape


pivot_table silently averaged the conflict: 644.9 | true value: 290.74


## F. Reproducible pipelines — functions, contracts, `.pipe`, saved outputs
Each function states an **input/output contract** and checks it with assertions; each is small enough to read
aloud on the projector. An **independent validation** compares the pipelined result against the step-by-step
result from section C.

In [25]:
def tidy_brent(raw_text):
    assert list(raw_text.columns) == ['observation_date', 'DCOILBRENTEU'], 'unexpected raw schema'
    miss = raw_text['DCOILBRENTEU'].eq('')
    out = pd.DataFrame({'date': pd.to_datetime(raw_text['observation_date'], format='%Y-%m-%d'),
                        'usd_per_barrel': pd.to_numeric(raw_text['DCOILBRENTEU'].mask(miss), errors='coerce')})
    assert not (~miss & out.usd_per_barrel.isna()).any(), 'malformed price token'
    assert out.usd_per_barrel.dropna().gt(0).all(), 'invalid price in this series'
    assert not out.date.duplicated().any(), 'dates must be unique'
    return out

def monthly_oil(frame):
    assert {'date', 'usd_per_barrel'}.issubset(frame.columns), 'typed input required'
    out = frame.groupby(frame.date.dt.to_period('M')).agg(
        mean_brent_usd=('usd_per_barrel', 'mean'), observed_days=('usd_per_barrel', 'count'))
    out = out.reset_index(names='period')
    assert not out.period.duplicated().any()
    return out

In [26]:
oil_monthly = raw.pipe(tidy_brent).pipe(monthly_oil)
staged = brent_monthly[['period', 'mean_brent_usd', 'observed_days']].reset_index(drop=True)
assert_frame_equal(oil_monthly.reset_index(drop=True), staged)      # independent validation
final = recent.merge(oil_monthly, on='period', how='left', validate='many_to_one', indicator=True)
assert len(final) == 36 and final._merge.eq('both').all()
print('Pipelined output matches the step-by-step result exactly; join cardinality re-validated.')

Pipelined output matches the step-by-step result exactly; join cardinality re-validated.


In [27]:
# Persist clean data, quarantine, issue log, the real joined table, and a run record.
clean.to_csv(OUT / 'lab_clean_prices.csv', index=False)
lab_quarantine.to_csv(OUT / 'lab_quarantine.csv', index=False)
issues.to_csv(OUT / 'lab_issues.csv', index=False)
final.drop(columns='_merge').to_csv(OUT / 'cpi_brent_monthly.csv', index=False)
record = {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__,
          'inputs': {'evidence_pack': str(PACK.relative_to(ROOT)), 'synthetic_lab': 'practice_prices_raw.csv'},
           'lab_accounting': {'raw_rows': len(lab), 'clean_keys': len(clean), 'observed': int(clean.value.count()),
                             'quarantined': len(lab_quarantine), 'issue_events': len(issues)},
           'real_join_rows': len(final), 'input_sha256': manifest}
(OUT / 'run_record.json').write_text(json.dumps(record, indent=2) + '\n')
print('Saved to', OUT.relative_to(ROOT), ':', sorted(p.name for p in OUT.glob('*')))

Saved to outputs : ['cpi_brent_monthly.csv', 'lab_clean_prices.csv', 'lab_issues.csv', 'lab_quarantine.csv', 'run_record.json']


## G. Return to the motivating investigation — two short recombinations
Nothing new is downloaded. Both cells read only the Lecture 4 V3 evidence pack, restate the grain of what they
read, and print the caveat next to the number — the habit we want at the end of any pipeline: recompute,
compare, and bound the interpretation.

In [28]:
# (1) iPhone pre-tax floor, recomputed from the same launch prices and same-day FX rates in the pack.
iphone = pd.read_csv(PACK / 'iphone_launch_comparison.csv')
floor = iphone.usd * iphone.pkr_per_usd                     # recomputed here, not copied from the file
assert np.allclose(floor, iphone.pkr_floor), 'persisted floor differs from recomputation'
chg = lambda s: 100 * (s.iloc[-1] / s.iloc[0] - 1)          # first vs last launch in the pack
print(f"2019→2026: USD {chg(iphone.usd):+.1f}%  ×  PKR/USD {chg(iphone.pkr_per_usd):+.1f}%"
      f"  →  pre-tax PKR floor {chg(floor):+.1f}%")
display(iphone[['year', 'model', 'usd', 'pkr_per_usd', 'pkr_floor']].round(2))
print('Grain: one row per launch year/model. Mechanical pre-tax floor — not a Pakistan retail price.')

2019→2026: USD +18.2%  ×  PKR/USD +77.6%  →  pre-tax PKR floor +110.0%


,year,model,usd,pkr_per_usd,pkr_floor
0,2019,iPhone 11 Pro Max,1099,156.08,171531.70
1,2021,iPhone 13 Pro Max,1099,169.08,185820.02
2,2023,iPhone 15 Pro Max,1199,291.76,349823.72
3,2026,iPhone 18 Pro Max,1299,277.25,360150.61


Grain: one row per launch year/model. Mechanical pre-tax floor — not a Pakistan retail price.


In [29]:
# (2) Fresh-fruit baskets — index changes over the two most recent months in the pack.
fruit = pd.read_csv(PACK / 'pbs_fresh_fruit_layers.csv')
levels = fruit.pivot(index='layer', columns='month', values='index_now')
levels['sep_vs_aug_pct'] = 100 * (levels['Sep 2026'] / levels['Aug 2026'] - 1)
display(levels.round(2))
print('Sep 2026 YoY (as published):', fruit.loc[fruit.month.eq('Sep 2026')]
      .set_index('layer').yoy_pct.round(1).to_dict())
print('CAVEAT: these are fresh-fruit BASKET indices (urban/rural CPI, wholesale WPI) — NOT apple-specific')
print('retail prices. Basket changes do not identify any individual apple price or confirm Rs100 → Rs300/kg.')

month,Aug 2026,Sep 2026,sep_vs_aug_pct
layer,,,
Rural retail (CPI),251.77,245.32,-2.56
Urban retail (CPI),253.85,253.04,-0.32
Wholesale (WPI),297.51,272.33,-8.46


Sep 2026 YoY (as published): {'Urban retail (CPI)': 14.8, 'Rural retail (CPI)': 7.0, 'Wholesale (WPI)': 1.0}
CAVEAT: these are fresh-fruit BASKET indices (urban/rural CPI, wholesale WPI) — NOT apple-specific
retail prices. Basket changes do not identify any individual apple price or confirm Rs100 → Rs300/kg.


## Conclusions and limits — what the transformed evidence now supports
- **Imported iPhone:** the pre-tax PKR floor is arithmetic on verified inputs; it is *not* a Pakistan retail
  price and excludes unverified PTA/FBR additions. USD price and exchange rate moved together — their effects
  multiply.
- **Local apples:** still **not fully tested**. The fresh-fruit *basket* index is not an apple price, and a CPI
  index level is not an inflation rate.
- **Oil ↔ CPI:** the validated monthly join aligns dates and units (USD/barrel vs index 2015-16=100). Shared
  months establish **alignment, not causality**; no pass-through share is estimated.
- **Reproducibility invariant:** re-running both notebooks from clean kernels must reproduce identical hashes,
  the 15 → 11/7/3/1 laboratory accounting, and the 36-row validated join.

In [30]:
print('Exit checks: (1) name the grain and key of each pack file; (2) why did oil need aggregation BEFORE',
      'the merge? (3) what does validate="many_to_one" promise, and what would break without it?')

Exit checks: (1) name the grain and key of each pack file; (2) why did oil need aggregation BEFORE the merge? (3) what does validate="many_to_one" promise, and what would break without it?
